# Cartesian diopters: conjugates, perturbation and acoustic excitation

This notebook accompanies [the model note](../docs/cartesian-diopters.md). The optical construction follows Silva-Lora and Torres, **JOSA A 37**, 1155–1165 (2020), [doi:10.1364/JOSAA.392795](https://doi.org/10.1364/JOSAA.392795), Eqs. (1), (3)–(11).

A single refracting surface is stigmatic for its chosen axial object/image pair and wavelength. That is distinct from the paper's complete two-surface singlet and from off-axis aplanatism.

The source notebook remains clean; execution and HTML export go to `artifacts/notebooks/`. No fluid simulation is launched by this notebook.

**Current result:** the optical target is verified. The acoustic stationary candidate remains mesh-sensitive (0.389 µm nominal ray RMS, 8.036 µm after refinement), with one growing mode in the nominal fixed-drive model. The example is not yet a validated stable stigmatic device. See [the research result](../docs/cartesian-results.md).

In [ ]:
from pathlib import Path
import csv, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dataclasses import replace
from acoustic_freeform.optics.cartesian import CartesianDiopter
from acoustic_freeform.single_interface.config import load_config, LensConfig
from acoustic_freeform.single_interface.surface import SurfaceSpace
from acoustic_freeform.single_interface.optics import trace_surface
from acoustic_freeform.single_interface.excitation import normal_shape_load
ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists()
plt.rcParams.update({"figure.dpi": 125, "axes.grid": True, "grid.alpha": 0.18})

## 1. The four optical parameters

Coordinates are measured from the **target vertex**; +z follows axial light propagation. A real object has $z_o<0$, a real image $z_i>0$; the subscript is the letter *o*. The incident index is $n_o$ and transmitted index $n_i$.

For the real-object/real-image branch,
$$n_o\sqrt{r^2+(z-z_o)^2}+n_i\sqrt{r^2+(z_i-z)^2}=n_o|z_o|+n_i|z_i|.$$

The Cartesian class also supports signed virtual conjugates and exact collimation limits. The fluid model below retains one liquid with air above it.

In [ ]:
n_o, z_o_m, n_i, z_i_m = 1.52, -0.050, 1.0, 0.020
diopter = CartesianDiopter(n_o, z_o_m, n_i, z_i_m)
r = np.linspace(0, 0.004, 501)
z = diopter.sag(r)
slope = diopter.slope(r, z)
parameters = diopter.form_parameters()
display(parameters)
print(f"Signed vertex curvature: {diopter.vertex_curvature_m_inv:.6f} m^-1")

## 2. Verify the paper's parametric formula and independent rays

The paper uses $\rho=\sqrt{r^2+z^2}$, **not** cylindrical radius $r$:
$$z(\rho)=\frac{(O+T\rho^2)\rho^2}{1+S\rho^2+\sqrt{1+(2S-O^2G)\rho^2}},\qquad r(\rho)=\sqrt{\rho^2-z(\rho)^2}.$$

We compare that formula against a continuation of the unsquared Fermat equation, then apply vector Snell refraction independently. The rays in the diagram below describe homogeneous optical media around a diopter; it is not an external source/window design.

In [ ]:
rp, zp = diopter.parametric(np.hypot(r,z))
outgoing = diopter.refract(r,z,slope)
intercept = r + (z_i_m-z)*outgoing[:,0]/outgoing[:,1]
print("Parametric / implicit maximum sag difference (m):", np.max(abs(zp-z)))
print("Maximum Snell ray intercept (m):", np.max(abs(intercept)))
print("Maximum Fermat residual (m):", np.max(abs(diopter.fermat(r,z))))
fig, ax = plt.subplots(figsize=(11,3.8),layout="constrained")
for sign in [-1,1]:
    ax.plot(z*1000, sign*r*1000, color="#008c95",lw=3)
    for radius in np.linspace(0.0003,0.003,10):
        height = float(diopter.sag(np.array(radius)))
        ax.plot([z_o_m*1000,height*1000,z_i_m*1000],[0,sign*radius*1000,0],color="#d29131",alpha=.7,lw=.8)
ax.scatter([z_o_m*1000,z_i_m*1000],[0,0],color="#27384a")
ax.set(xlabel="z from target vertex (mm)",ylabel="Transverse coordinate (mm)",title="Exact finite-conjugate Cartesian diopter: prescribed incident wave in nₒ")
plt.show()

## 3. Shape family, mounting and fill volume

The mounted target is $h_\star(r)=z(r)-z(a)$, with $a=4$ mm. The mounting fixes the required volume. Each curve below therefore represents a **separately filled experiment**, not instantaneous tuning at fixed fill.

The perturbation is relative to the nonlinear undriven equilibrium at that same fill: $\delta h=h_\star-h_0$.

In [ ]:
base = load_config(ROOT / "configs/lenses/noa61_cartesian_50_20.toml")
fig, axs = plt.subplots(1,2,figsize=(11,4),layout="constrained")
family=[]
for zo in [None,-0.100,-0.050,-0.025]:
    cfg=replace(base,object_distance_m=zo)
    space=SurfaceSpace(cfg)
    target,volume=space.target()
    initial=space.equilibrium(np.zeros(space.count),volume)
    h=cfg.radius_m*space.evaluate(target,r/cfg.radius_m)
    delta=cfg.radius_m*space.evaluate(target-initial,r/cfg.radius_m)
    label="collimated" if zo is None else f"zₒ = {zo*1000:g} mm"
    axs[0].plot(r*1000,h*1000,label=label)
    axs[1].plot(r*1000,delta*1e6,label=label)
    fill=np.pi*cfg.radius_m**2*cfg.depth_m+2*np.pi*cfg.radius_m**3*volume
    family.append((label,fill*1e9,h[0]*1000))
axs[0].set(xlabel="Radius (mm)",ylabel="Height above rim (mm)",title="Cartesian targets, image at +20 mm")
axs[1].set(xlabel="Radius (mm)",ylabel="Required perturbation (µm)",title="Change from each fill's unforced equilibrium")
axs[0].legend(); plt.show()
display(Markdown("| Object | Fill (µL) | Target cap (mm) |\n|---|---:|---:|\n"+"\n".join(f"| {a} | {b:.6f} | {c:.6f} |" for a,b,c in family)))

## 4. Required traction and why the array design is an inverse problem

Using the outward normal into air,
$$\Pi_\star=\sigma\kappa+\rho gh_\star-\lambda,\qquad
\kappa=-\left(\frac{h''}{(1+h'^2)^{3/2}}+\frac{h'}{r\sqrt{1+h'^2}}\right).$$

The constant pressure $\lambda$ enforces conserved volume. Acoustic traction is
$$\Pi(r;w,h)=\frac{\rho}{4}\left|\sum_jw_jV_{n,j}(r;h)\right|^2.$$
It includes coherent cross terms and changes with the cavity geometry. The computed fluid state must therefore be checked after the array fit; a frozen target fit is insufficient.

In [ ]:
space=SurfaceSpace(base)
target,volume=space.target()
initial=space.equilibrium(np.zeros(space.count),volume)
fig,ax=plt.subplots(figsize=(8,3.4),layout="constrained")
for state,label in [(initial,"Unforced equilibrium"),(target,"Cartesian target")]:
    load=normal_shape_load(space,state,r)
    load-=np.trapezoid(load*r,r)/np.trapezoid(r,r)
    ax.plot(r*1000,load,label=label)
ax.set(xlabel="Radius (mm)",ylabel="Shape load minus area mean (Pa)",title="Spatial force required at fixed fill; pressure gauge removed")
ax.legend();plt.show()

## 5. Restricted-drive transient: an actual forward result

The following cells read the saved coupled simulation. The array has 16 independent coherent rows; all 16 sectors of a row share its drive. The finite-conjugate incident wavefront is prescribed inside the resin. External illumination through the flat window is not designed.

The final holding excitation uses
$$v_{n,j}(z,t)=e_j(z)\operatorname{Re}[w_j e^{-i2\pi ft}].$$
The file gives peak wall velocities, not voltages. A common phase offset is arbitrary. The preceding time program is needed to approach the held shape in the recorded experiment.

This first 1.35 MHz run is a documented limitation: its ~48 µm ray RMS is not a stigmatic result. Section 8 contains the separate joint stationary design with a larger provisional drive allowance.

In [ ]:
result=ROOT/"artifacts/studies/S01-single-interface/cartesian-50-20"
report=json.loads((result/"report.json").read_text())
trajectory=np.load(result/"trajectory.npz")
excitation=json.loads((result/"excitation/definition.json").read_text())
with (result/"excitation/hold-drive.csv").open() as f:
    rows=list(csv.DictReader(f))
display(Markdown("| Row from bottom | Peak velocity (mm/s) | Relative phase (deg) | Peak displacement (nm) |\n|---:|---:|---:|---:|\n"+"\n".join(f"| {x['row_from_bottom_1based']} | {float(x['velocity_peak_m_s'])*1000:.4f} | {float(x['phase_relative_to_row1_deg']):.3f} | {float(x['displacement_peak_m'])*1e9:.4f} |" for x in rows)))
final=report['final']
print(f"Computed figure RMS: {final['shape_rms_to_target_m']*1e6:.6f} µm")
print(f"Computed geometric RMS ray radius at fixed image plane: {final['rms_spot_at_target_m']*1e6:.6f} µm")
print(f"Peak cavity pressure: {final['peak_acoustic_pressure_pa']/1e6:.6f} MPa")
print(f"Source power: {final['source_power_w']*1000:.6f} mW")
print(f"Maximum slow liquid velocity at final time: {final['maximum_fluid_speed_m_s']*1e6:.6f} µm/s")
print(f"Conserved fill: {report['liquid_volume_m3']*1e9:.6f} µL")

In [ ]:
drive=trajectory['drive_m_s'][-1]
phase=np.degrees(np.angle(drive*drive[0].conjugate()))
fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout="constrained")
rows=np.arange(1,len(drive)+1)
axs[0].bar(rows,abs(drive)*1000,color="#009c9c")
axs[1].plot(rows,phase,"o-",color="#b37b2a")
axs[0].set(xlabel="Array row, bottom to top",ylabel="Peak normal wall velocity (mm/s)",title="Computed holding amplitudes")
axs[1].set(xlabel="Array row, bottom to top",ylabel="Phase relative to row 1 (degrees)",title=f"Coherent phases at {base.frequency_hz/1e6:g} MHz")
plt.show()
fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout="constrained")
for state,label in [(trajectory['initial'],"Unforced"),(trajectory['target'],"Optical target"),(trajectory['coefficients'][-1],"Computed fluid")]:
    axs[0].plot(r*1000,base.radius_m*space.evaluate(state,r/base.radius_m)*1000,label=label)
axs[0].set(xlabel="Radius (mm)",ylabel="Height above rim (mm)",title="Actual driven surface")
axs[0].legend()
axs[1].semilogy(trajectory['times_s'],[x['rms_spot_at_target_m']*1e6 for x in report['history']],color="#009c9c")
axs[1].set(xlabel="Physical fluid time (s)",ylabel="Geometric RMS ray radius (µm)",title="Fixed image plane; fixed incident wavefront")
plt.show()

## 6. Stigmatic is not automatically aplanatic

For the same on-axis surface, examine $\sin\alpha_o/\sin\alpha_i$ across the pupil. A changing ratio does not satisfy the Abbe sine condition. The axial equal-OPL result alone does not establish off-axis coma correction or achromatism.

In [ ]:
pupil=np.linspace(.0001,.003,301)
h=diopter.sag(pupil)
incoming=diopter.incident_direction(pupil,h)
outgoing=diopter.refract(pupil,h,diopter.slope(pupil,h))
ratio=incoming[:,0]/outgoing[:,0]
print("Relative range of sine ratio:", np.ptp(ratio)/abs(ratio.mean()))
fig,ax=plt.subplots(figsize=(7,3),layout="constrained")
ax.plot(pupil*1000,ratio)
ax.set(xlabel="Pupil radius (mm)",ylabel="sin αₒ / sin αᵢ",title="Single-diopter sine ratio")
plt.show()
verification=result/"spatial-convergence.json"
if verification.exists():
    checks=json.loads(verification.read_text())
    for x in checks:
        print(f"P{x['order']}, mesh {x['mesh']}, modes {x['modes']}: ray RMS {x['rms_spot_at_target_m']*1e6:.6f} µm, shape change {x['surface_change_rms_m']*1e9:.4f} nm")

## 7. Change a conjugate and compute a new excitation

Edit `[diopter]` in a copy of `configs/lenses/noa61_cartesian_50_20.toml`. Keep the material and acoustic assumptions consistent. Use a new result directory:

```bash
uv run lenslab simulate configs/lenses/your_diopter.toml --out artifacts/your_diopter
uv run lenslab excitation artifacts/your_diopter
uv run lenslab render artifacts/your_diopter
```

For optical-only exploration, instantiate `CartesianDiopter(n_o,z_o_m,n_i,z_i_m)` and evaluate `sag`, `parametric`, `fermat`, and `refract`. The full fluid apparatus currently supports liquid-to-air real-conjugate focusing; general optical solutions are not promises of acoustic reachability.

These remain monochromatic geometric-optics and nominal-acoustics calculations. Streaming, heating, unsteady fluid inertia, calibrated transducer response, curing, diffraction and external illumination require further models.

## 8. Joint stationary candidate: refinement and stability

The exact surface and array field must be solved together. The stationary algorithm linearizes optics and capillarity around the current computed surface, then independently checks force balance with fixed drives.

This example uses **1.8 MHz**, P4 acoustics, and a provisional wall-speed limit of **1.28 m/s**. This is not a calibrated transducer specification. The output is a stationary solution; optimization iterations are not time evolution.

The design-mesh ray RMS is 0.389 µm, but re-solving the identical drive on a finer mesh gives **8.036 µm**. Thus the nominal optical precision is not established. The fixed-drive stability calculation also finds one growing mode; that calculation has not yet been spatially refined. The figures below show both optical results rather than accepting the attractive design-mesh ray fan alone.

In [ ]:
stationary=ROOT/"artifacts/studies/S01-single-interface/cartesian-50-20-stationary-p4"
sr=json.loads((stationary/"report.json").read_text())
sa=np.load(stationary/"stationary.npz")
sconfig=LensConfig(**sr["configuration"])
surface=SurfaceSpace(sconfig)
print("Requested diopter:",sr["diopter"])
print(f"Nominal design-mesh ray RMS: {sr['optics']['rms_spot_at_target_m']*1e6:.6f} µm")
print(f"Nominal design-mesh optical path RMS: {sr['optics']['opd_rms_m']*1e9:.6f} nm")
print(f"Largest peak wall velocity: {sr['max_wall_velocity_peak_m_s']:.6f} m/s")
print(f"Largest peak displacement: {sr['max_wall_displacement_peak_m']*1e9:.6f} nm")
print(f"Peak cavity pressure: {sr['peak_cavity_pressure_pa']/1e6:.6f} MPa")
if 'stability' in sr:
    print("Largest real fixed-drive growth rate (1/s):",sr['stability']['largest_real_growth_rate_s_inv'])
    print("Number of locally unstable modes:",sr['stability']['unstable_modes'])
with (stationary/'hold-drive.csv').open() as f:
    holding=list(csv.DictReader(f))
display(Markdown("| Row from bottom | Peak velocity (m/s) | Relative phase (deg) | Peak displacement (nm) |\n|---:|---:|---:|---:|\n"+"\n".join(f"| {x['row_from_bottom_1based']} | {float(x['velocity_peak_m_s']):.6f} | {float(x['phase_relative_to_row1_deg']):.3f} | {float(x['displacement_peak_m'])*1e9:.4f} |" for x in holding)))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout="constrained")
for state,label,color in [(sa['target'],'Exact optical target','#d6a044'),(sa['coefficients'],'Nominal stationary candidate','#009c9c')]:
    axs[0].plot(r*1000,sconfig.radius_m*surface.evaluate(state,r/sconfig.radius_m)*1000,label=label,color=color)
actual=trace_surface(surface,sa['coefficients'],count=2001)
axs[1].plot(actual['pupil_r_m']*1000,actual['target_spot_r_m']*1e6,color='#009c9c',label='Design mesh')
checks=json.loads((stationary/'spatial-convergence.json').read_text())
last=checks[-1]
nr,nz=last['mesh']; modes=last['surface_modes']
fineconfig=replace(sconfig,acoustic_order=last['order'],mesh_radial=nr,mesh_vertical=nz,surface_modes=modes)
finespace=SurfaceSpace(fineconfig)
finedata=np.load(stationary/f"refined-P{last['order']}-{nr}-{nz}-{modes}.npz")
fineray=trace_surface(finespace,finedata['coefficients'],count=2001)
axs[0].plot(r*1000,fineconfig.radius_m*finespace.evaluate(finedata['coefficients'],r/fineconfig.radius_m)*1000,'--',color='#b4593e',label='Same drive, refined')
axs[1].plot(fineray['pupil_r_m']*1000,fineray['target_spot_r_m']*1e6,color='#b4593e',label='Same drive, refined')
axs[0].set(xlabel='Radius (mm)',ylabel='Height above rim (mm)',title='Nominal and refined stationary surfaces')
axs[0].legend(fontsize=8)
axs[1].set(xlabel='Pupil radius (mm)',ylabel='Ray intercept at specified image (µm)',title='Fixed-drive refinement changes the optical result')
axs[1].legend(fontsize=8)
plt.show()
display(Markdown('| Acoustic order | Mesh | Surface modes | Ray RMS (µm) | OPD RMS (nm) | Surface change (nm) |\n|---:|---|---:|---:|---:|---:|\n'+'\n'.join(f"| {q['order']} | {q['mesh']} | {q['surface_modes']} | {q['ray_rms_at_target_m']*1e6:.6f} | {q['opd_rms_m']*1e9:.3f} | {q['surface_change_rms_m']*1e9:.3f} |" for q in checks)))

To reproduce the recorded stationary optimization branch:

```bash
uv run lenslab stationary configs/lenses/noa61_cartesian_50_20_stationary.toml \
  --seed configs/initialization/cartesian_50_20.json --starts 1 \
  --out artifacts/another_stationary --stability
uv run lenslab verify artifacts/another_stationary
uv run lenslab render artifacts/another_stationary
```

The seed is the preceding P3 drive; the optimizer computes the final P4 candidate. The standalone PyVista scene is explicitly stationary and displays the refinement and stability limitations. The earlier time viewer displays its separately integrated physical trajectory.

**What remains:** isolate geometry/acoustic/surface truncation error with fixed-drive studies; then find a stable branch or demonstrate a feedback-controlled trajectory. The four optical parameters determine a stigmatic target, while the apparatus and coupled dynamics determine whether it can be produced and maintained. Current excitations are inspectable research candidates, not a calibrated fabrication recipe.